# C2 — Dự báo Màu sắc & SKU Bán chạy Q2/2026
**DATA EXPLORERS 2026 | Xe đạp Thống Nhất**

Phân tích và dự báo:
- Top màu sắc bán chạy
- Top SKU (mẫu xe) bán chạy / chậm bán
- Dự báo xu hướng Q2/2026 theo màu/nhóm sản phẩm

**Thứ tự chạy:** tuần tự từ trên xuống (Run All)


## 1. Thư viện & Cài đặt

In [ ]:
# thư viện
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import pmdarima as pm
from statsmodels.tsa.stattools import adfuller

## 2. Tải & Làm sạch Dữ liệu

In [ ]:
# tải dữ liệu giao dịch đầy đủ (22,357 đơn hàng)
df_full = pd.read_csv('../data_raw/fact_sales_full.csv', parse_dates=['order_date'])
print(f'Shape: {df_full.shape}')
print(f'Khoảng thời gian: {df_full.order_date.min().strftime("%m/%Y")} – {df_full.order_date.max().strftime("%m/%Y")}')
df_full.head()

In [ ]:
# kiểm tra missing
print('Missing values (%) :')
miss = df_full.isnull().mean()*100
print(miss[miss > 0].round(2))

# thống kê cơ bản
print(f'\nSố đơn hàng    : {len(df_full):,}')
print(f'Số sản phẩm (SKU): {df_full.product_code.nunique():,}')
print(f'Số màu sắc      : {df_full.color.nunique():,}')
print(f'Số nhóm SP      : {df_full.group_code.nunique():,}')
print(f'Số tỉnh/thành   : {df_full.province_name.nunique():,}')

In [ ]:
# bỏ dòng không xác định được sản phẩm
df_clean = df_full.dropna(subset=['product_name', 'color', 'group_code']).copy()
print(f'Sau clean: {len(df_clean):,} đơn hàng ({len(df_clean)/len(df_full)*100:.1f}%)')

# thêm cột tháng
df_clean['month'] = df_clean['order_date'].dt.to_period('M')
df_clean['year']  = df_clean['order_date'].dt.year

## 3. EDA — Phân tích Màu sắc

In [ ]:
# top 15 màu theo doanh số
top_colors = (df_clean.groupby('color')['line_total']
              .sum().sort_values(ascending=False).head(15))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

ax1.barh(range(len(top_colors)), top_colors.values/1e9, alpha=0.85)
ax1.set_yticks(range(len(top_colors)))
ax1.set_yticklabels(top_colors.index, fontsize=9)
ax1.set_xlabel('Doanh số (tỷ VND)')
ax1.set_title('Top 15 Màu theo Doanh số', fontsize=12, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='x')

# top 15 màu theo số lượng
top_colors_qty = (df_clean.groupby('color')['quantity']
                  .sum().sort_values(ascending=False).head(15))
ax2.barh(range(len(top_colors_qty)), top_colors_qty.values, alpha=0.85)
ax2.set_yticks(range(len(top_colors_qty)))
ax2.set_yticklabels(top_colors_qty.index, fontsize=9)
ax2.set_xlabel('Số lượng (chiếc)')
ax2.set_title('Top 15 Màu theo Số lượng', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='x')

fig.suptitle('Phân tích Màu sắc — Xe đạp Thống Nhất', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

In [ ]:
# doanh số theo nhóm sản phẩm
group_rev = (df_clean.groupby('group_name')['line_total']
             .sum().sort_values(ascending=False))

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(range(len(group_rev)), group_rev.values/1e9, alpha=0.85)
ax.set_xticks(range(len(group_rev)))
ax.set_xticklabels(group_rev.index, rotation=20, ha='right', fontsize=10)
ax.set_ylabel('Doanh số (tỷ VND)')
ax.set_title('Doanh số theo Nhóm sản phẩm', fontsize=13, fontweight='bold')
for i, v in enumerate(group_rev.values):
    ax.text(i, v/1e9 + 0.2, f'{v/1e9:.1f}T', ha='center', fontsize=9, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')
fig.tight_layout()
plt.show()

## 4. EDA — Phân tích SKU

In [ ]:
# top 20 SKU bán chạy nhất
top_sku = (df_clean.groupby('product_name')['quantity']
           .sum().sort_values(ascending=False).head(20))

fig, ax = plt.subplots(figsize=(12, 8))
ax.barh(range(len(top_sku)), top_sku.values, alpha=0.85)
ax.set_yticks(range(len(top_sku)))
ax.set_yticklabels([n[:45] for n in top_sku.index], fontsize=8)
ax.set_xlabel('Tổng số lượng bán (chiếc)')
ax.set_title('Top 20 SKU Bán chạy nhất — Thống Nhất Bike', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
fig.tight_layout()
plt.show()

In [ ]:
# SKU chậm bán (dưới phân vị 20%)
sku_avg = df_clean.groupby('product_name')['quantity'].mean()
slow_sku = sku_avg[sku_avg < sku_avg.quantile(0.20)].sort_values().head(20)

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(range(len(slow_sku)), slow_sku.values, alpha=0.85)
ax.set_yticks(range(len(slow_sku)))
ax.set_yticklabels([n[:45] for n in slow_sku.index], fontsize=8)
ax.set_xlabel('Trung bình số lượng/tháng (chiếc)')
ax.set_title('Top 20 SKU Chậm bán nhất — Cần xem xét', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
fig.tight_layout()
plt.show()

In [ ]:
# xu hướng doanh số theo tháng và nhóm sản phẩm
pivot = (df_clean.groupby(['month','group_name'])['line_total']
         .sum().unstack(fill_value=0) / 1e9)
pivot.index = pivot.index.to_timestamp()

fig, ax = plt.subplots(figsize=(14, 6))
for col in pivot.columns:
    ax.plot(pivot.index, pivot[col], marker='o', linewidth=2, label=col)
ax.set_title('Doanh số theo Nhóm sản phẩm — Theo tháng', fontsize=13, fontweight='bold')
ax.set_ylabel('Doanh số (tỷ VND)')
ax.set_xlabel('Tháng')
ax.legend(fontsize=9, loc='upper left')
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 5. Dự báo xu hướng Màu sắc Q2/2026 (SARIMAX)

In [ ]:
# chuẩn bị time series theo màu top 5
top5_colors = top_colors.head(5).index.tolist()

color_monthly = (df_clean[df_clean.color.isin(top5_colors)]
                 .groupby(['month','color'])['quantity']
                 .sum().unstack(fill_value=0))
color_monthly.index = color_monthly.index.to_timestamp()
color_monthly.tail()

In [ ]:
# ADF test cho từng màu
from statsmodels.tsa.stattools import adfuller
for col in top5_colors:
    pv = adfuller(color_monthly[col].dropna())[1]
    status = 'DỪNG' if pv < 0.05 else 'KHÔNG DỪNG'
    print(f'  {col:<20}: p={pv:.3f} → {status}')

In [ ]:
# SARIMAX dự báo 3 tháng Q2/2026 cho từng màu top 5
forecasts_color = {}
for col in top5_colors:
    try:
        model_c = pm.auto_arima(
            color_monthly[col].values,
            seasonal=True, m=3,
            stepwise=True, suppress_warnings=True, error_action='ignore'
        )
        pred = model_c.predict(n_periods=3)
        forecasts_color[col] = np.maximum(pred, 0)  # không âm
        print(f'✅ {col}: {[f"{v:.0f}" for v in pred]}')
    except Exception as e:
        print(f'❌ {col}: {e}')

In [ ]:
# biểu đồ dự báo màu sắc Q2/2026
months_q2 = pd.date_range('2026-04-01', periods=3, freq='MS')
months_label = ['T4/2026', 'T5/2026', 'T6/2026']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# lịch sử
ax = axes[0]
for col in top5_colors:
    ax.plot(color_monthly.index, color_monthly[col], marker='o', linewidth=2, label=col)
ax.set_title('Lịch sử bán hàng — Top 5 Màu', fontsize=12, fontweight='bold')
ax.set_ylabel('Số lượng (chiếc)')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# dự báo Q2
ax = axes[1]
x = np.arange(3)
width = 0.15
for i, col in enumerate(top5_colors):
    if col in forecasts_color:
        ax.bar(x + i*width, forecasts_color[col], width, label=col, alpha=0.85)
ax.set_xticks(x + width*2)
ax.set_xticklabels(months_label)
ax.set_ylabel('Dự báo số lượng (chiếc)')
ax.set_title('Dự báo Q2/2026 — Top 5 Màu', fontsize=12, fontweight='bold')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis='y')

fig.suptitle('Phân tích & Dự báo Màu sắc — Thống Nhất Bike', fontsize=13, fontweight='bold')
fig.tight_layout()
plt.show()

## 6. Tăng trưởng SKU — Phát hiện rủi ro

In [ ]:
# so sánh doanh số 2 tháng gần nhất
recent = df_clean[df_clean.order_date >= '2025-10-01'].copy()
prev   = df_clean[(df_clean.order_date >= '2025-07-01') &
                   (df_clean.order_date < '2025-10-01')].copy()

rev_recent = recent.groupby('product_name')['line_total'].sum().rename('recent')
rev_prev   = prev.groupby('product_name')['line_total'].sum().rename('prev')

growth = pd.concat([rev_recent, rev_prev], axis=1).dropna()
growth['growth_rate'] = (growth.recent - growth.prev) / (growth.prev + 1)

slow = growth[growth.growth_rate < -0.1].sort_values('growth_rate').head(15)
print(f'SKU nguy cơ bán chậm (tăng trưởng âm): {len(slow)} SKU')

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(slow['product_name'].str[:40], slow['growth_rate']*100, alpha=0.85)
ax.axvline(0, linestyle='--', alpha=0.5)
ax.set_xlabel('Tăng trưởng (%)')
ax.set_title('SKU có Tốc độ Tăng trưởng Âm — Cần chú ý', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
fig.tight_layout()
plt.show()

## 7. Xuất Kết quả

In [ ]:
# tổng hợp dự báo màu
rows = []
for col, vals in forecasts_color.items():
    for i, (month, val) in enumerate(zip(months_label, vals)):
        rows.append({'color': col, 'tháng': month, 'qty_forecast': int(val)})

df_color_forecast = pd.DataFrame(rows)
df_color_forecast.to_csv('../Forecasting Product/Ensemble/predictions_color_sku.csv', index=False)
print('✅ Đã lưu predictions_color_sku.csv')
df_color_forecast

In [ ]:
# tóm tắt
print('=' * 50)
print('  KẾT QUẢ C2 — MÀU SẮC & SKU Q2/2026')
print('=' * 50)
print(f'\n→ Top 3 màu bán chạy: {top_colors.head(3).index.tolist()}')
print(f'→ Top 3 SKU bán chạy: {top_sku.head(3).index.tolist()}')
print(f'→ Số SKU chậm bán: {len(slow_sku)} SKU cần xem xét')
print(f'→ Số SKU nguy cơ Q2: {len(slow)} SKU tăng trưởng âm')
print('=' * 50)